In [0]:
from pyspark.sql import functions as F

In [0]:
source_path = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "customers/2026-08-17/customers.csv"
)

customer_df = (
    spark.read.option("header","true")\
                        .option("inferSchema","true")\
                            .csv(source_path)
)

display(customer_df.limit(10))

### Adding Metadata
- for future refrence
- to trace the origin of data

In [0]:
bronze_df = (
    customer_df
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_source_date",
        F.to_date(
            F.regexp_extract(
                F.col("_metadata.file_path"),
                r"/customers/(\d{4}-\d{2}-\d{2})/",
                1
            )
        )
    )
)

display(bronze_df.limit(10))

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dbx_fintech_data_platform.bronze;

In [0]:
bronze_table = "dbx_fintech_data_platform.bronze.customers"

(
    bronze_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_table)
)

In [0]:
%sql
SELECT
    _source_date,
    COUNT(*) AS record_count
FROM dbx_fintech_data_platform.bronze.customers
GROUP BY _source_date
ORDER BY _source_date;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dbx_fintech_data_platform.metadata;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbx_fintech_data_platform.metadata.ingestion_log (
    pipeline_name STRING,
    source_file STRING,
    source_date DATE,
    target_table STRING,
    status STRING,
    rows_processed BIGINT,
    started_at TIMESTAMP,
    completed_at TIMESTAMP,
    error_message STRING
)
USING DELTA;

In [0]:
source_file = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/customers/2026-08-18/customers.csv"
)

existing_count = spark.sql(f"""
    SELECT COUNT(*) AS cnt
    FROM dbx_fintech_data_platform.metadata.ingestion_log
    WHERE source_file = '{source_file}'
      AND status = 'SUCCESS'
""").collect()[0]["cnt"]

print("Already processed:", existing_count > 0)

In [0]:
source_path = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "customers/2026-08-18/customers.csv"
)

customer_18_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(source_path)
)

In [0]:
bronze_18_df = (
    customer_18_df
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_source_date",
        F.to_date(
            F.regexp_extract(
                F.col("_metadata.file_path"),
                r"/customers/(\d{4}-\d{2}-\d{2})/",
                1
            )
        )
    )
)

In [0]:
print("Rows:", bronze_18_df.count())
display(bronze_18_df.limit(5))

In [0]:
(
    bronze_18_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("dbx_fintech_data_platform.bronze.customers")
)

In [0]:
from datetime import datetime, date

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType,
    DateType
)

log_schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("source_file", StringType(), True),
    StructField("source_date", DateType(), True),
    StructField("target_table", StringType(), True),
    StructField("status", StringType(), True),
    StructField("rows_processed", LongType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("completed_at", TimestampType(), True),
    StructField("error_message", StringType(), True)
])

log_data = [(
    "customer_bronze_ingestion",
    source_path,
    date(2026, 8, 18),
    "dbx_fintech_data_platform.bronze.customers",
    "SUCCESS",
    100000,
    datetime.now(),
    datetime.now(),
    None
)]

log_df = spark.createDataFrame(
    log_data,
    schema=log_schema
)

log_df.write.mode("append").saveAsTable(
    "dbx_fintech_data_platform.metadata.ingestion_log"
)

In [0]:
%sql
SELECT *
FROM dbx_fintech_data_platform.metadata.ingestion_log;

In [0]:
existing_count = spark.sql(f"""
    SELECT COUNT(*) AS cnt
    FROM dbx_fintech_data_platform.metadata.ingestion_log
    WHERE source_file = '{source_path}'
      AND status = 'SUCCESS'
""").collect()[0]["cnt"]

if existing_count > 0:
    print("SKIPPED: Source file already processed.")
else:
    print("NEW FILE: Ready for ingestion.")

In [0]:
%sql
SELECT
    _source_date,
    COUNT(*) AS record_count
FROM dbx_fintech_data_platform.bronze.customers
GROUP BY _source_date
ORDER BY _source_date;